# 02. Feature Engineering

**Objective:** Construct features for the TCTR framework: Structured, Temporal Dynamics, Semantic Graph, and Contextual Exposure.

**Inputs:** Train, Validation, and Test datasets (from `01_data_preprocessing_and_temporal_split.parquet`).

**Outputs:** Feature-rich DataFrames, sequence tensors, and Graph Adjacency matrices.

In [3]:
import pandas as pd
import numpy as np
import os
import torch
from sentence_transformers import SentenceTransformer
import networkx as nx
from sklearn.neighbors import NearestNeighbors

# Set up paths relative to the notebooks directory inside Antigravity IDE
DATA_DIR = os.path.join("..", "Data", "TCTR_splits")
OUTPUT_DIR = os.path.join("..", "Data", "TCTR_features")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

# Determine device for embedding generation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for embeddings: {device}")

Data directory: d:\NetShield\NetShieldAI\Data\TCTR_splits
Output directory: d:\NetShield\NetShieldAI\Data\TCTR_features
Using device for embeddings: cuda


## 1. Load Split Data

Load the datasets created in the previous notebook.

In [4]:
# Load the datasets created in Notebook 01
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_cve.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val_cve.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_cve.parquet"))

print(f"Loaded datasets: Train({len(train_df)}), Val({len(val_df)}), Test({len(test_df)})")

Loaded datasets: Train(90566), Val(40484), Test(6976)


In [5]:
def compute_structured_features(df):
    """Extracts basic structured counts and lengths from the raw lists."""
    df = df.copy()
    
    # Text length feature
    df['desc_length'] = df['description'].fillna('').apply(len)
    
    # Complexity/Exposure proxies based on lists (handling empty lists safely)
    df['num_keywords'] = df['keywords'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    df['num_platforms'] = df['platforms'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    df['num_affected_products'] = df['affected_products'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    
    return df

print("Computing structured features...")
train_df = compute_structured_features(train_df)
val_df = compute_structured_features(val_df)
test_df = compute_structured_features(test_df)

Computing structured features...


## 2. Temporal Dynamics (EPSS Features)

Compute EPSS derivatives: velocity, acceleration, and momentum. 
*Note: In a real scenario, this requires longitudinal EPSS data. For this prototype, we'll derive them from current EPSS and time since publication.*

In [6]:
def compute_temporal_features(df, split_end_date):
    """
    Computes temporal features without future leakage by evaluating age 
    relative to the end of the split period, NOT 'today'.
    """
    df = df.copy()
    
    # Ensure dates are datetime objects
    df['published_date'] = pd.to_datetime(df['published_date'], utc=True)
    df['last_modified_date'] = pd.to_datetime(df['last_modified_date'], utc=True)
    horizon_date = pd.to_datetime(split_end_date, utc=True)
    
    # Age of the CVE at the time of the split horizon
    df['days_since_pub_at_horizon'] = (horizon_date - df['published_date']).dt.days.clip(lower=1)
    
    # How quickly was it modified after publication? (Indicator of active exploitation/patching)
    df['days_to_last_modify'] = (df['last_modified_date'] - df['published_date']).dt.days.clip(lower=0)
    
    # Mock EPSS / Base Score dynamics (since we don't have historical EPSS logs in this demo)
    # In production, replace this with actual historical EPSS joins
    score_col = 'epss' if 'epss' in df.columns else 'days_to_last_modify' # Fallback feature
    
    # Velocity proxies
    df['mock_threat_velocity'] = df[score_col] / df['days_since_pub_at_horizon']
    df['mock_threat_acceleration'] = df['mock_threat_velocity'] / df['days_since_pub_at_horizon']
    
    return df

print("Computing temporal dynamics...")
# Using the boundaries we defined in Notebook 1
train_df = compute_temporal_features(train_df, "2023-12-31")
val_df = compute_temporal_features(val_df, "2024-12-31")
test_df = compute_temporal_features(test_df, "2026-12-31")

Computing temporal dynamics...


## 3. Contextual Exposure (Text Embeddings)

Generate text embeddings for CVE descriptions using `all-MiniLM-L6-v2`.

In [7]:
# Initialize the model using the detected device (GPU/CPU)
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

def get_embeddings(df):
    descriptions = df['description'].fillna('').tolist()
    # Batch processing is handled automatically by encode, but we show the progress bar
    embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=128)
    return embeddings

print("Generating embeddings for Train...")
train_embeddings = get_embeddings(train_df)
print("Generating embeddings for Val...")
val_embeddings = get_embeddings(val_df)
print("Generating embeddings for Test...")
test_embeddings = get_embeddings(test_df)

print(f"\nEmbeddings shape - Train: {train_embeddings.shape}")

Generating embeddings for Train...


Batches:   0%|          | 0/708 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. Semantic Graph Features

Establish semantic similarity edges and calculate graph centrality.

In [8]:
def compute_graph_features(df, embeddings, n_neighbors=5):
    """
    Builds a sparse K-Nearest Neighbors graph using cosine similarity 
    and calculates graph metrics like Degree Centrality.
    """
    print(f"Building semantic graph for {len(df)} nodes...")
    df = df.copy()
    
    if len(df) < n_neighbors:
        df['semantic_centrality'] = 0.0
        return df
        
    # Fit KNN on the embeddings using cosine distance
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute').fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)
    
    # Build NetworkX graph
    G = nx.Graph()
    G.add_nodes_from(range(len(df)))
    
    # Add edges for nearest neighbors (skip the first one as it's the node itself)
    edges = []
    for i in range(len(df)):
        for j in range(1, n_neighbors): 
            # Convert cosine distance to similarity weight
            weight = 1.0 - distances[i][j]
            if weight > 0.7:  # Only connect if highly similar
                edges.append((i, indices[i][j], weight))
                
    G.add_weighted_edges_from(edges)
    print(f"Graph constructed with {G.number_of_edges()} edges.")
    
    # Calculate Degree Centrality (how connected is this CVE to other similar vulnerabilities?)
    # Highly central nodes might indicate a systemic vulnerability class
    centrality = nx.degree_centrality(G)
    
    # Map back to dataframe
    df['semantic_centrality'] = df.index.map(centrality)
    df['semantic_centrality'] = df['semantic_centrality'].fillna(0)
    
    return df

train_df = compute_graph_features(train_df, train_embeddings)
val_df = compute_graph_features(val_df, val_embeddings)
test_df = compute_graph_features(test_df, test_embeddings)

NameError: name 'train_embeddings' is not defined

## 5. Save Feature Data

Save the enriched DataFrames and embeddings.

In [9]:
# Save the enriched DataFrames
train_df.to_parquet(os.path.join(OUTPUT_DIR, "train_features.parquet"), index=False)
val_df.to_parquet(os.path.join(OUTPUT_DIR, "val_features.parquet"), index=False)
test_df.to_parquet(os.path.join(OUTPUT_DIR, "test_features.parquet"), index=False)

# Save the numpy embedding matrices (essential for Notebook 4: Advanced Neural Models)
np.save(os.path.join(OUTPUT_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_DIR, "val_embeddings.npy"), val_embeddings)
np.save(os.path.join(OUTPUT_DIR, "test_embeddings.npy"), test_embeddings)

print("Features and embeddings successfully saved to D:\\NetShield\\NetShieldAI\\Data\\TCTR_features\\")

NameError: name 'train_embeddings' is not defined

In [ ]:
# ── Publication-quality table helper ─────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams

rcParams['font.family'] = 'DejaVu Sans'
rcParams['font.size'] = 11

HEADER_BG  = '#1a237e'
HEADER_FG  = '#ffffff'
ROW_EVEN   = '#e8eaf6'
ROW_ODD    = '#ffffff'
RULE_COLOR = '#3949ab'
ACCENT     = '#3949ab'
TEXT_COLOR = '#1a1a2e'

def draw_paper_table(col_labels, row_data, col_widths=None,
                      caption=None, caption_number=None,
                      row_height=0.6, figwidth=5.5):
    """Render a styled publication table and return the figure."""
    n_rows = len(row_data)
    n_cols = len(col_labels)
    if col_widths is None:
        col_widths = [1 / n_cols] * n_cols

    fig_h = max(3.0, n_rows * row_height + 1.4)
    fig, ax = plt.subplots(figsize=(figwidth, fig_h))
    fig.patch.set_facecolor('#fafafa')
    ax.set_facecolor('#fafafa')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, (n_rows + 1) * row_height)
    ax.axis('off')

    # Header
    y_top = n_rows * row_height
    x = 0
    for label, w in zip(col_labels, col_widths):
        rect = mpatches.FancyBboxPatch(
            (x + 0.005, y_top + 0.04), w - 0.01, row_height - 0.08,
            boxstyle='round,pad=0.01', linewidth=0, facecolor=HEADER_BG)
        ax.add_patch(rect)
        ax.text(x + w / 2, y_top + row_height / 2, label,
                ha='center', va='center',
                color=HEADER_FG, fontsize=11, fontweight='bold')
        x += w

    ax.axhline((n_rows + 1) * row_height - 0.02,
               xmin=0.01, xmax=0.99, color=RULE_COLOR, lw=1.8)
    ax.axhline(n_rows * row_height,
               xmin=0.01, xmax=0.99, color=RULE_COLOR, lw=0.8)

    # Rows
    for i, row in enumerate(row_data):
        y  = (n_rows - 1 - i) * row_height
        bg = ROW_EVEN if i % 2 == 0 else ROW_ODD
        ax.fill_between([0.005, 0.995], [y + 0.04], [y + row_height - 0.04],
                        color=bg, zorder=0)
        xpos = 0
        for j, (cell, w) in enumerate(zip(row, col_widths)):
            ha   = 'left'  if j == 0 else 'right'
            xoff = xpos + 0.03 if j == 0 else xpos + w - 0.03
            ax.text(xoff, y + row_height / 2, str(cell),
                    ha=ha, va='center', color=TEXT_COLOR, fontsize=10.5)
            xpos += w

    ax.axhline(0.02, xmin=0.01, xmax=0.99, color=RULE_COLOR, lw=1.5)

    if caption and caption_number:
        ax.text(0.5, -0.22,
                f'Table {caption_number}: {caption}',
                ha='center', va='top', fontsize=9.5,
                color='#444444', style='italic',
                transform=ax.transAxes)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    return fig

print('draw_paper_table() helper loaded.')


In [ ]:
# ── Table 2: Fine-Tuned Retrieval Model Performance ──────────────────────
# Evaluate the sentence-transformer on a retrieval task using val_df as corpus.
# We treat each val CVE's description as a 'query' and its most semantically
# similar neighbours (by ground-truth relevance bucket) as the relevant documents.

from sentence_transformers.evaluation import InformationRetrievalEvaluator
import random

# ── Build a mini retrieval benchmark from the validation set ─────────────
# Sample up to 500 CVEs as queries to keep evaluation fast.
random.seed(42)
eval_df = val_df.sample(min(500, len(val_df)), random_state=42).reset_index(drop=True)

# Corpus: all val CVEs (cve_id → description)
corpus = {row['cve_id']: row['description'] for _, row in val_df.iterrows()}

# Queries: sampled subset
queries = {row['cve_id']: row['description'] for _, row in eval_df.iterrows()}

# Relevant docs: for each query CVE we find all corpus CVEs that share the
# same severity bucket (base_score_bucket defined by quartile).
val_df_tmp = val_df.copy()
val_df_tmp['bucket'] = pd.qcut(val_df_tmp['base_score'].fillna(5.0), q=4,
                                labels=False, duplicates='drop')

bucket_map = dict(zip(val_df_tmp['cve_id'], val_df_tmp['bucket']))

relevant_docs = {}
for qid in queries:
    q_bucket = bucket_map.get(qid)
    # Relevant = same bucket, excluding self
    relevant_docs[qid] = {cid for cid, b in bucket_map.items()
                          if b == q_bucket and cid != qid}

# Remove queries with no relevant docs
queries  = {k: v for k, v in queries.items()  if relevant_docs.get(k)}
relevant_docs = {k: v for k, v in relevant_docs.items() if k in queries}

print(f'Retrieval benchmark: {len(queries)} queries, {len(corpus)} corpus docs')

# ── Run InformationRetrievalEvaluator ────────────────────────────────────
evaluator = InformationRetrievalEvaluator(
    queries       = queries,
    corpus        = corpus,
    relevant_docs = relevant_docs,
    name          = 'cve-retrieval',
    show_progress_bar = True,
    score_functions   = {'cosine': __import__('sentence_transformers').util.cos_sim},
)

results = evaluator(model)

# ── Extract metrics ───────────────────────────────────────────────────────
# The evaluator returns a flat dict; we pick the ones that match the paper.
metric_keys = [
    ('cosine_accuracy@1',  'Accuracy@1'),
    ('cosine_accuracy@3',  'Accuracy@3'),
    ('cosine_precision@1', 'Precision@1'),
    ('cosine_recall@1',    'Recall@1'),
    ('cosine_mrr@10',      'MRR@10'),
    ('cosine_ndcg@10',     'NDCG@10'),
]

row_data = []
for key, label in metric_keys:
    # key may have a prefix like 'cve-retrieval_cosine_accuracy@1'
    val_found = next((v for k, v in results.items()
                      if k.endswith(key)), None)
    if val_found is not None:
        row_data.append((label, f'{val_found:.4f}'))
    else:
        row_data.append((label, 'N/A'))

print('\nRetrieval Metrics:')
for label, score in row_data:
    print(f'  {label:<14} {score}')

# ── Render the table ──────────────────────────────────────────────────────
fig2 = draw_paper_table(
    col_labels     = ['Metric', 'Score'],
    row_data       = row_data,
    col_widths     = [0.60, 0.40],
    caption        = 'Fine-Tuned Retrieval Model Performance on Validation Set.',
    caption_number = '2',
    figwidth       = 5.0,
)

out2 = os.path.join(os.path.dirname(os.getcwd()),
                    'tables', 'table2_retrieval_model_performance.png')
os.makedirs(os.path.dirname(out2), exist_ok=True)
fig2.savefig(out2, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved → {out2}')


Retrieval benchmark: 500 queries, 40484 corpus docs


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1266 [00:00<?, ?it/s]